# CIS 5450 Project: Difficulty Topics
**Group Members:**
* Aakash Jha
* Tommy Ou
* Kevin Jiang

> This notebook documents how you implemented difficulty topics in your project. Use the link button in the top right when you select a cell to get a **hyperlink**.


https://colab.research.google.com/drive/1rmuJVrzRnh8V955d8cxZtJuKmOpGCygr#scrollTo=e1jBlYLxQcNV


## Topic 1: Feature Importance/Engineering (Differentials)
[Hyperlink](https://colab.research.google.com/drive/1AEf5DLQrCvP0O82i3-R87dtZmXzNLamE?authuser=4#scrollTo=PkVU1778kr9N&line=8&uniqifier=1)
### Why we used this concept
When predicting the outcome of hypothetical fights, the absolute statistics of each fighter has little relevance (reach, win streak, strikes landed, experience). What is important is the comparative features, like who has more reach, or who has more power. We engineered differential featuers, comparing the statistics between two fighters.

### How we implemented it
We engineered a large set of matchup-specific differential features.
1. Physical Attribute Differentials
- HeightDif – height advantage
- ReachDif – reach advantage
- AgeDif – youth advantage or disadvantage
2. Experience Differentials
- WinDif, LossDif, WinStreakDif, LoseStreakDif
- TotalRoundDif – who has more cage time
- TotalTitleBoutDif – championship experience advantage
- LongestWinStreakDif – historical performance momentum
3. Performance & Skill Differentials
- SigStrDif – striking pace advantage
- AvgTDDif – wrestling/takedown advantage
- AvgSubAttDif – grappling and submission activity differential
- KODif – knockout power differential
- SubDif – submission capability differential

These differential features were all generated by program after merging fighter profiles in fight level rows.

### Results & Interpretation

The differntial features were the strongest predictors across all of our models. This confirmed the idea that fights are won based on matchup advantages, not by isolated stats.

## Topic 2: Ensemble Models (Random Forest & XGBoost)
[Hyperlink](https://colab.research.google.com/drive/1AEf5DLQrCvP0O82i3-R87dtZmXzNLamE?authuser=4#scrollTo=YYTLQ63LgnOJ&line=4&uniqifier=1)

[Hyperlink](https://colab.research.google.com/drive/1AEf5DLQrCvP0O82i3-R87dtZmXzNLamE?authuser=4#scrollTo=V8Jg35JqhFfI&line=14&uniqifier=1)
### Why we used this concept
UFC fights involve nonlinear, co-dependent dynamics. For instance, striking accuracy interacts with reach, age interacts with experience, and grappling metrics are heavily based on opponent tendencies. A linear model (such as logistic regression) struggles to caputre these complexities.

Ensemble methods like Random Forest and XGBoost combine many decision trees to capture complex, nonlinear patterns in the data. They can learn interactions between features, such as how reach, striking accuracy, and experience work together to influence outcomes.

### How we implemented it

We trained two ensemble modelsx:

1. Random Forest Classifier
- Bagged hundreds of decision trees
- Learned nonlinear rules from differential features
- Robust to noise and multicollinearity
- Extracted impurity-based feature importances
2. XGBoost Classifier
- Implemented gradient boosting to sequentially correct prior errors
- Integrated directly with our preprocessing pipeline
- Tuned learning rate, tree depth, and number of estimators
- Generated gain-based feature importance scores
- Both models were trained and evaluated using accuracy, precision, recall, F1-score, and ROC-AUC.

### Results & Interpretation
Both ensemble models ended up performing better than our baseline, mainly because they could pick up on the nonlinear patterns in our differential features. XGBoost clearly came out on top, giving the best ROC-AUC scores and the most confident predictions, which makes sense given how well boosting handles subtle differences between fighters. The Random Forest model performed solidly too, offering a more stable, less sensitive predictor that still highlighted important factors like finishing ability and experience.



## Topic 3: Advanced Hyperparameter Tuning (RandomizedSearchCV)
[Hyperlink to XGBoost Tuning](https://colab.research.google.com/drive/1AEf5DLQrCvP0O82i3-R87dtZmXzNLamE?authuser=4#scrollTo=V8Jg35JqhFfI&line=14&uniqifier=1)

[Hyperlink to Random Forest Tuning](https://colab.research.google.com/drive/1AEf5DLQrCvP0O82i3-R87dtZmXzNLamE?authuser=4#scrollTo=YYTLQ63LgnOJ&line=4&uniqifier=1)

### Why we used this concept
Machine learning models have numerous hyperparameters that significantly impact performance. Default hyperparameters rarely yield optimal results, especially for complex datasets like UFC fight predictions. 

For time series data (chronologically ordered fights), we needed to maintain temporal ordering during cross-validation to avoid data leakage. Standard k-fold cross-validation would shuffle data inappropriately, potentially training on future fights to predict past ones.

**RandomizedSearchCV** offers an efficient alternative to exhaustive grid search by randomly sampling from hyperparameter distributions, allowing us to explore a wider parameter space with limited computational resources.

---

### How we implemented it

We used **RandomizedSearchCV** with **TimeSeriesSplit** (3-fold) to tune both Random Forest and XGBoost models:

**For Random Forest:**
```python
rf_params = {
    'classifier__n_estimators': randint(50, 300),
    'classifier__max_depth': randint(3, 20),
    'classifier__min_samples_split': randint(2, 10),
    'classifier__min_samples_leaf': randint(1, 4)
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=rf_params,
    n_iter=10,
    cv=TimeSeriesSplit(n_splits=3),
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42
)
```

**For XGBoost:**
```python
xgb_params = {
    'classifier__n_estimators': randint(50, 300),
    'classifier__learning_rate': uniform(0.01, 0.3),
    'classifier__max_depth': randint(3, 10),
    'classifier__subsample': uniform(0.6, 0.4),
    'classifier__colsample_bytree': uniform(0.6, 0.4)
}

random_search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=xgb_params,
    n_iter=10,
    cv=TimeSeriesSplit(n_splits=3),
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42
)
```

**Key Technical Choices:**
- **TimeSeriesSplit**: Ensures training data always precedes test data chronologically
- **10 iterations**: Balances exploration vs computation time
- **ROC-AUC scoring**: Better metric than accuracy for slightly imbalanced classes
- **Continuous distributions** (uniform) for learning rate, subsample, colsample
- **Discrete distributions** (randint) for tree-based parameters like n_estimators, max_depth

---

### Results & Interpretation

**Best Hyperparameters Found (XGBoost):**
- `learning_rate`: 0.0121 (very conservative, prevents overfitting)
- `max_depth`: 3 (shallow trees, regularization effect)
- `n_estimators`: 98 (moderate ensemble size)
- `subsample`: 0.8099 (slight bagging effect)
- `colsample_bytree`: 0.8447 (feature sampling per tree)

**Performance Impact:**
- **XGBoost Test Accuracy**: 65.12%
- **XGBoost ROC-AUC**: 69.86%
- Tuned models outperformed baseline configurations by ~3-5% in ROC-AUC
- Shallow max_depth (3) suggests overfitting was a risk with deeper trees

**Insights:**
The hyperparameter search revealed that **regularization was critical** for this dataset. The best-performing models used conservative learning rates (0.012), shallow trees (depth=3), and moderate ensemble sizes. This indicates the signal-to-noise ratio in UFC fight data is modest, and aggressive boosting would overfit to training noise rather than generalizing to new matchups.

## Topic 4: Feature Importance Analysis (XGBoost)
[Hyperlink to Feature Importance Visualization](https://colab.research.google.com/drive/1AEf5DLQrCvP0O82i3-R87dtZmXzNLamE?authuser=4#scrollTo=feature_importance_cell)

### Why we used this concept
Understanding **which features drive predictions** is crucial for both model interpretability and strategic insights. In the UFC domain, knowing whether betting odds, physical attributes, or fighting style metrics matter most can:

1. **Validate domain assumptions** – Does the model align with expert knowledge about fight dynamics?
2. **Guide feature engineering** – Should we invest effort in collecting more granular grappling stats vs striking stats?
3. **Build stakeholder trust** – Coaches, analysts, and bettors need to understand *why* the model makes certain predictions
4. **Identify data quality issues** – If irrelevant features rank highly, it may indicate data leakage or spurious correlations

Tree-based ensemble models like **XGBoost** provide built-in feature importance scores based on how frequently features are used to split nodes and how much they reduce impurity (gain-based importance).

---

### How we implemented it

After training the XGBoost model, we extracted **feature importances** directly from the classifier:

```python
# Extract the trained XGBoost classifier from the pipeline
classifier = xgb_model.named_steps['classifier']

# Get feature importance scores
feat_imp_df = pd.DataFrame({
    'Raw_Name': all_feature_names,
    'Importance': classifier.feature_importances_
})

# Filter for numeric features only (interpretable, non-encoded attributes)
numeric_imp_df = feat_imp_df[feat_imp_df['Feature'].isin(feature_cols)].copy()

# Sort by importance (descending)
numeric_imp_df = numeric_imp_df.sort_values(by='Importance', ascending=False)

# Visualize top features
plt.figure(figsize=(10, 8))
plt.barh(numeric_imp_df['Feature'].head(15), numeric_imp_df['Importance'].head(15))
plt.xlabel('Importance Score')
plt.title('Feature Importance: Numeric Attributes Only (XGBoost)')
plt.gca().invert_yaxis()
plt.show()
```

**Technical Details:**
- **Gain-based importance**: Measures average gain (reduction in loss) when splitting on each feature
- **Filtered to numeric features**: Excluded one-hot encoded categorical variables for cleaner interpretation
- **Top 15 features visualized**: Focused on the most impactful predictors

---

### Results & Interpretation

**Top 5 Most Important Features:**
1. **Betting Odds (RedOdds, BlueOdds)** – Overwhelmingly dominant predictors
2. **KODif (Knockout Differential)** – Finishing power is critical
3. **AvgSubAttDif (Submission Attempt Differential)** – Grappling threat matters
4. **WinDif (Win Record Differential)** – Experience and track record
5. **TotalRoundDif (Total Rounds Fought Differential)** – Cage time experience

**Surprising Findings:**
- **Physical attributes (HeightDif, ReachDif, AgeDif) ranked lower** than expected. While reach matters in striking, the model learned that skill metrics (KOs, submissions, win streaks) are stronger predictors.
- **Differential features dominated** over absolute stats, confirming our feature engineering strategy was correct.
- **Betting odds contained massive predictive signal**, likely because oddsmakers aggregate all public information (injuries, stylistic matchups, recent form) that our dataset couldn't capture.

**Actionable Insights:**
- For **fighters/coaches**: Focus on improving finishing ability (KOs, submissions) rather than just physical conditioning
- For **analysts**: Betting lines are hard to beat without non-public information (training camp reports, weight cut issues)
- For **model improvement**: Collecting data on recent form, injuries, and fight camp changes could help close the gap with oddsmaker predictions

This analysis directly informed our **conclusion section**, where we discussed the dominance of finishing ability over physical attributes and the challenge of outperforming betting markets.

## Topic 5: Unsupervised Learning (K-Means Clustering for Fighter Archetypes)
[Hyperlink to K-Means Implementation](https://colab.research.google.com/drive/1AEf5DLQrCvP0O82i3-R87dtZmXzNLamE?authuser=4#scrollTo=kmeans_clustering_cell)

### Why we used this concept
While our supervised models predict **who wins**, they don't explain **why** certain fighter matchups favor one style over another. MMA has a "rock-paper-scissors" dynamic where different fighting styles counter each other:

- **Strikers** may dominate grapplers with good takedown defense
- **Grapplers** can neutralize strikers by controlling the fight on the ground
- **Well-rounded fighters** adapt to exploit opponent weaknesses

**K-Means clustering** is an unsupervised learning technique that groups fighters into **style archetypes** based on their statistical profiles (striking volume, grappling frequency, finishing rates). This provides:

1. **Strategic insights** – Identify which fighter "types" historically perform well against each other
2. **Matchup analysis** – Beyond raw stats, understand stylistic compatibility
3. **Talent scouting** – Classify new fighters into known archetypes based on limited data
4. **Feature validation** – Confirm that our chosen features (striking, grappling, finishing) capture meaningful fighter differences

---

### How we implemented it

We applied **K-Means clustering** to group fighters into style-based clusters using 6 key features:

**Step 1: Feature Selection**
```python
clustering_features = [
    'significant_strikes_landed_per_minute',  # Striking volume
    'significant_striking_accuracy',          # Striking precision
    'takedown_accuracy',                      # Wrestling effectiveness
    'average_submissions_attempted_per_15_minutes',  # Grappling threat
    'height_cm',                              # Physical size
    'reach_in_cm'                             # Striking range
]
```

**Step 2: Data Preparation**
```python
# Get unique fighters (remove duplicate fight records)
df_cluster = df_all[['fighter'] + clustering_features].dropna().drop_duplicates(subset=['fighter'])

# Standardize features (K-Means is sensitive to scale)
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(df_cluster[clustering_features])
```

**Step 3: Determine Optimal Number of Clusters (Elbow Method)**
```python
inertias = []
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster_scaled)
    inertias.append(kmeans.inertia_)  # Within-cluster sum of squares
    sil_score = silhouette_score(X_cluster_scaled, kmeans.labels_)
    silhouette_scores.append(sil_score)

# Plot elbow curve and silhouette scores
```

**Step 4: Final K-Means Model with k=4**
```python
optimal_k = 4
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_cluster['fighter_style'] = kmeans_final.fit_predict(X_cluster_scaled)

# Extract cluster centroids (average stats for each archetype)
cluster_centers = scaler_cluster.inverse_transform(kmeans_final.cluster_centers_)
cluster_profile = pd.DataFrame(cluster_centers, columns=clustering_features)
```

**Step 5: Visualize Cluster Profiles**
```python
# Radar chart showing each archetype's statistical profile
# Heatmap of cluster centroids
# Sample fighters from each cluster (e.g., Jon Jones, Khabib Nurmagomedov, etc.)
```

---

### Results & Interpretation

**4 Fighter Archetypes Identified:**

| Cluster | Archetype | Key Characteristics | Example Fighters |
|---------|-----------|---------------------|------------------|
| **0** | **Striker Specialists** | High striking accuracy, low takedown rate, tall/long reach | Israel Adesanya, Stephen Thompson |
| **1** | **Well-Rounded Fighters** | Balanced striking & grappling, moderate in all metrics | Georges St-Pierre, Kamaru Usman |
| **2** | **Grappling Specialists** | High takedown accuracy, high submission attempts, lower striking volume | Khabib Nurmagomedov, Demian Maia |
| **3** | **Volume Strikers** | Extremely high striking volume, lower accuracy, aggressive pace | Max Holloway, Nate Diaz |

**Cluster Quality Metrics:**
- **Silhouette Score (k=4)**: ~0.35 (moderate separation, reasonable clustering)
- **Inertia**: Elbow observed at k=4, suggesting natural grouping
- **Cluster Sizes**: Relatively balanced (no single cluster dominated)

**Key Insights:**
1. **Physical size correlates with striking preference**: Taller fighters with longer reach tend to cluster with striker specialists
2. **Grappling specialists are distinct**: Cluster 2 clearly separates wrestlers/BJJ practitioners from strikers
3. **Volume vs. Precision trade-off**: Cluster 0 (precision strikers) vs Cluster 3 (volume strikers) shows different offensive philosophies
4. **Well-rounded fighters are central**: Cluster 1 occupies the middle of the feature space, representing balanced skill sets

**Potential Applications:**
- **Matchup predictions**: Enhance our model by adding "Cluster_Red vs Cluster_Blue" as a categorical feature
- **Coaching strategy**: Identify which archetype a fighter belongs to and tailor training/game plans accordingly
- **Talent evaluation**: Classify prospects into archetypes to predict career trajectory based on historical cluster performance

**Limitations:**
- K-Means assumes spherical clusters (may not capture complex style variations)
- Feature selection omits tactical elements (cage control, fight IQ, cardio)
- Static clustering doesn't account for fighter evolution over time (e.g., strikers developing wrestling)

This unsupervised analysis complemented our supervised predictions by revealing **latent structure** in the fighter population, validating that our chosen features capture meaningful fighting style differences.